## 1. Ejercicio A — Precio de Vivienda

Dataset: 14 propiedades, 4 features (Área, Habitaciones, Antigüedad, Distancia).

In [ ]:
DATA = np.array([
    [ 95, 4, 13, 12.5, 171_600], [123, 2, 14,  8.0, 216_200],
    [118, 2, 22, 15.0, 196_900], [ 96, 4,  8,  5.5, 190_200],
    [182, 4, 19,  9.0, 305_700], [ 86, 4,  5,  3.5, 180_100],
    [ 63, 2,  7, 18.0, 133_600], [193, 2,  3,  4.0, 320_000],
    [155, 2, 29, 22.0, 242_700], [128, 2,  3,  6.5, 216_800],
    [195, 5, 18,  7.5, 357_400], [115, 5, 22, 20.0, 219_800],
    [105, 3, 10,  2.0, 198_500], [140, 3,  6, 14.0, 241_300],
], dtype=float)
pd.DataFrame(DATA, columns=['Area', 'Hab', 'Antig', 'Dist', 'Precio'])

### 1.1 Gradiente Descendiente a mano — 4 iteraciones, lr=0.1

In [ ]:
x = np.array([0.95, 1.23, 1.18])
y_p = np.array([1.71, 2.16, 1.96])
w0, w1 = 0.0, 1.0
lr = 0.1; m = len(x)
rows = []
for it in range(1, 5):
    yhat = w0 + w1*x; err = yhat - y_p
    mse = (err**2).mean()
    g0 = (2/m)*err.sum(); g1 = (2/m)*(err*x).sum()
    rows.append([it, w0, w1, mse, g0, g1])
    w0 -= lr*g0; w1 -= lr*g1
pd.DataFrame(rows, columns=['iter','w0','w1','MSE','dw0','dw1']).round(4)

### 1.2 Ecuación normal + GD computacional (dataset completo)

In [ ]:
X_raw = DATA[:, :4]; y_full = DATA[:, 4].reshape(-1, 1)
mu = X_raw.mean(axis=0); sd = X_raw.std(axis=0)
X_norm = (X_raw - mu) / sd
Xb = np.hstack([np.ones((14, 1)), X_norm])

# Ecuación normal
w_ne = np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y_full

# GD computacional
w_gd = np.zeros((5, 1))
lr, epochs = 0.01, 5000
for ep in range(epochs):
    err = Xb @ w_gd - y_full
    w_gd -= lr * (2/14) * (Xb.T @ err)
    if ep % 500 == 0:
        print(f'  ep {ep:>4}: MSE = {(err**2).mean():,.0f}')

feat = ['Intercepto', 'Area', 'Hab', 'Antig', 'Dist']
pd.DataFrame({'NE': w_ne.flatten(), 'GD': w_gd.flatten()}, index=feat).round(2)

In [ ]:
# Predicción: 120 m², 3 hab, 10 años, 7 km
nueva = np.array([[120, 3, 10, 7]], dtype=float)
nueva_n = (nueva - mu) / sd
pred = float((np.hstack([[[1.0]], nueva_n]) @ w_gd).item())
print(f'Predicción: ${pred:,.0f}')

### 1.3 Regresión Logística (Premium: precio > $200,000)

In [ ]:
y_bin = (DATA[:, 4] > 200_000).astype(int)
print(f'Premium={y_bin.sum()}, Estándar={(1-y_bin).sum()}')

def sigmoid(z): return 1/(1+np.exp(-np.clip(z, -500, 500)))

def fit_logreg(X, y, lr=0.01, epochs=2000):
    Xb = np.hstack([np.ones((X.shape[0],1)), X])
    w = np.zeros(Xb.shape[1]); m = len(X)
    for _ in range(epochs):
        w -= lr * (Xb.T @ (sigmoid(Xb@w) - y)) / m
    return w

def acc(X, y, w):
    Xb = np.hstack([np.ones((X.shape[0],1)), X])
    return ((sigmoid(Xb@w) >= 0.5).astype(int) == y).mean()

names = ['Área', 'Habitaciones', 'Antigüedad', 'Distancia']
rows = []
for i, name in enumerate(names):
    Xi = X_norm[:, [i]]
    w = fit_logreg(Xi, y_bin)
    rows.append([name, w[0], w[1], acc(Xi, y_bin, w)])
pd.DataFrame(rows, columns=['Feature', 'w0', 'wi', 'Accuracy']).round(4)

In [ ]:
# Multivariado
w_full = fit_logreg(X_norm, y_bin, lr=0.01, epochs=3000)
print(f'Pesos: {np.round(w_full, 3)}')
print(f'Accuracy multivariada: {acc(X_norm, y_bin, w_full):.3f}')

# Confusión
yhat = (sigmoid(np.hstack([np.ones((14,1)), X_norm]) @ w_full) >= 0.5).astype(int)
print(f'\nMatriz de Confusión: TN={((yhat==0)&(y_bin==0)).sum()}, '
      f'FP={((yhat==1)&(y_bin==0)).sum()}, '
      f'FN={((yhat==0)&(y_bin==1)).sum()}, '
      f'TP={((yhat==1)&(y_bin==1)).sum()}')

# Predicción: 175 m², 4 hab, 8 años, 6 km
nueva = np.array([[175, 4, 8, 6]], dtype=float)
nueva_n = (nueva - mu)/sd
p = float(sigmoid(np.hstack([[[1.0]], nueva_n]) @ w_full).item())
print(f'\nP(PREMIUM | 175m², 4hab, 8a, 6km) = {p:.3f} → {"PREMIUM" if p>=0.5 else "ESTÁNDAR"}')

In [ ]:
# Frontera de decisión (Área vs Antigüedad)
Xi = X_norm[:, [0, 2]]
w2 = fit_logreg(Xi, y_bin, lr=0.01, epochs=3000)
aa, bb = np.meshgrid(np.linspace(Xi[:,0].min()-0.5, Xi[:,0].max()+0.5, 200),
                      np.linspace(Xi[:,1].min()-0.5, Xi[:,1].max()+0.5, 200))
grid = np.c_[aa.ravel(), bb.ravel()]
pp = sigmoid(np.hstack([np.ones((grid.shape[0],1)), grid]) @ w2).reshape(aa.shape)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(aa, bb, pp, levels=20, cmap='RdBu_r', alpha=0.6)
ax.contour(aa, bb, pp, levels=[0.5], colors='black', linewidths=2)
ax.scatter(Xi[y_bin==0,0], Xi[y_bin==0,1], c='#4C72B0', s=80, edgecolor='white', label='Estándar')
ax.scatter(Xi[y_bin==1,0], Xi[y_bin==1,1], c='#C44E52', s=80, edgecolor='white', label='Premium')
ax.set_xlabel('Área (z-score)'); ax.set_ylabel('Antigüedad (z-score)')
ax.set_title('Frontera de decisión — Logística'); ax.legend()
plt.tight_layout(); plt.show()